In [0]:
%sql

-- ---------------------------------------------
-- BRONZE LAYER: STORAGE ACCESS CONFIGURATION
-- ---------------------------------------------
-- Purpose:
-- This step creates an external location in Azure Data Lake Storage (ADLS)
-- to securely connect Databricks with the raw data container.
--
-- Why this is needed:
-- - Enables Databricks to read/write data in Azure Storage
-- - Separates compute (Databricks) from storage (ADLS)
-- - Supports Medallion Architecture best practice
--
-- Security:
-- - Uses a pre-configured storage credential for authentication
-- - Avoids hardcoding secrets in code


create external location IF not exists dea_course_ext_dl_spr_ridesshare
URL 'abfss://ridesproject@deacourseextdlspr.dfs.core.windows.net/'
WITH (storage credential dea_course_ext_sc)

In [0]:
%sql

-- ---------------------------------------------
-- LAKEHOUSE SETUP: CATALOG CREATION
-- ---------------------------------------------
-- Purpose:
-- This statement creates a Unity Catalog to organize all data assets
-- (tables, schemas, views) for the rides data lakehouse project.
--
-- Why this is needed:
-- - Provides a top-level namespace for data governance
-- - Enables centralized access control and metadata management
-- - Supports Medallion Architecture (bronze/silver/gold schemas)
--
-- Storage:
-- - Uses a managed storage location in Azure Data Lake Storage (ADLS)
-- - All managed tables in this catalog will store data in this location
--
-- Benefit:
-- - Ensures better data governance, security, and separation of environments

CREATE CATALOG IF NOT EXISTS rides_catalog
MANAGED LOCATION 'abfss://ridesproject@deacourseextdlspr.dfs.core.windows.net/'
COMMENT "This is the catalog for the rides Data Lakehouse"

In [0]:
%sql
-- ---------------------------------------------
-- BRONZE LAYER: SCHEMA CREATION
-- ---------------------------------------------
-- Purpose:
-- This schema represents the Bronze layer in the Medallion Architecture.
-- It stores raw, unprocessed data ingested directly from Azure Data Lake.
--
-- Why this is needed:
-- - Separates raw data from cleaned and aggregated datasets
-- - Enables structured layering of data (Bronze → Silver → Gold)
-- - Supports data lineage and governance under Unity Catalog
--
-- Data Characteristics:
-- - Raw ingestion layer
-- - Minimal or no transformations applied
-- - May contain nulls, duplicates, or inconsistent values

create schema if not exists rides_catalog.bronze;


In [0]:
# ---------------------------------------------
# BRONZE LAYER: DATA INGESTION FROM AZURE DATALAKE
# ---------------------------------------------
# Purpose:
# This step reads raw ride data from Azure Data Lake Storage (ADLS)
# into a Spark DataFrame for further processing in the Medallion pipeline.
#
# Why this is needed:
# - Acts as the entry point for all raw data in the pipeline
# - Loads data from external storage (Azure Data Lake) into Databricks
# - Prepares dataset for Bronze table creation (raw layer storage)
#
# Data Characteristics:
# - CSV file format
# - Header row is present
# - Schema is inferred automatically (not enforced manually)


df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://ridesproject@deacourseextdlspr.dfs.core.windows.net/landing/rides_data.csv")

In [0]:
# SILVER LAYER: DATA ENRICHMENT (INGESTION METADATA)
# ---------------------------------------------
# Purpose:
# This step adds metadata to track when each record was ingested
# into the system.
#
# Why this is needed:
# - Enables data lineage tracking
# - Helps with debugging and auditing
# - Supports incremental processing in real pipelines
# - Standard practice in Medallion Architecture Silver layer
#
# Transformation:
# - Adds a new column 'ingestion_time' with current timestamp
#   indicating when the record entered the Silver processing stage


from pyspark.sql.functions import current_timestamp

bronze_df = df.withColumn("ingestion_time", current_timestamp())


In [0]:
# ---------------------------------------------
# BRONZE LAYER: WRITE RAW DATA TO DELTA TABLE
# ---------------------------------------------
# Purpose:
# This step persists the raw ingested data into the Bronze layer
# as a Delta table in Unity Catalog.
#
# Why this is needed:
# - Stores raw data in a structured, queryable format
# - Enables time travel and versioning via Delta Lake
# - Serves as the foundation for downstream Silver transformations
#
# Data Characteristics:
# - Raw (minimal transformations applied)
# - Includes ingestion metadata (ingestion_time)
# - Stored in Delta format for reliability and performance

bronze_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true").saveAsTable("rides_catalog.bronze.rides_raw")

In [0]:
%sql
-- ---------------------------------------------
-- BRONZE LAYER: DATA VALIDATION / EXPLORATION
-- ---------------------------------------------
-- Purpose:
-- This query is used to validate that raw data has been successfully
-- ingested into the Bronze Delta table.
--
-- Why this is important:
-- - Confirms ingestion pipeline worked correctly
-- - Allows quick inspection of raw data structure and quality
-- - Helps identify issues before Silver layer transformations
--
-- Note:
-- - Data in Bronze layer is raw and may contain nulls, duplicates,
--   or inconsistent values (expected behavior in Medallion architecture)


select * from rides_catalog.bronze.rides_raw;